# Tutorial 03 — Co-sponsorship Network Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyusik-yang/assembly-explorer/blob/main/tutorials/03_network_analysis.ipynb)

**Research question**: Who legislates together, and do co-sponsorship networks reveal cross-party coalitions?

Legislative co-sponsorship networks are a standard tool in political science to measure
lawmaker preferences, ideological proximity, and inter-party collaboration
(Fowler 2006; Kirkland & Gross 2014).

**What you'll learn:**
- Build an **undirected co-sponsorship graph** from Korean Assembly bill records
- Compute **network centrality** measures (degree, betweenness, closeness)
- Detect **communities** (clusters of closely collaborating legislators)
- Visualize the network with party-colored nodes
- Export node-level and edge-list datasets for further analysis

**Prerequisites:** Tutorial 01 (API key + AssemblyAPI class); networkx.

---

In [ ]:
!pip install httpx pandas plotly networkx -q

In [ ]:
import os

try:
    from google.colab import userdata
    API_KEY = userdata.get('ASSEMBLY_API_KEY')
except Exception:
    API_KEY = os.getenv('ASSEMBLY_API_KEY', '')

if not API_KEY:
    API_KEY = input('Enter your ASSEMBLY_API_KEY: ').strip()

print('API key loaded:', bool(API_KEY))

In [ ]:
import httpx
import time

class AssemblyAPI:
    """Synchronous client for the Korean National Assembly Open API."""

    BASE = 'https://open.assembly.go.kr/portal/openapi'

    def __init__(self, api_key: str):
        self.key = api_key

    def _get(self, endpoint: str, **params) -> tuple[list[dict], int]:
        p = {'KEY': self.key, 'Type': 'json',
             **{k: v for k, v in params.items() if v is not None}}
        r = httpx.get(f'{self.BASE}/{endpoint}', params=p, timeout=30)
        r.raise_for_status()
        body = r.json().get(endpoint, [])
        if not body:
            return [], 0
        head  = body[0].get('head', [])
        total = int(head[0].get('list_total_count', 0)) if head else 0
        code  = (head[1].get('RESULT', {}) if len(head) > 1 else {}).get('CODE', '')
        if code == 'INFO-200':
            return [], 0
        rows = body[1].get('row', []) if len(body) > 1 else []
        return (rows if isinstance(rows, list) else [rows]), total

    def search_bills(self, age, bill_name=None, proposer=None,
                     committee=None, proc_result=None,
                     dt_from=None, dt_to=None, page=1, page_size=100):
        return self._get('nzmimeepazxkubdpn', AGE=age, BILL_NAME=bill_name,
                         PROPOSER=proposer, PROC_RESULT=proc_result,
                         COMMITTEE=committee, STR_DT=dt_from, END_DT=dt_to,
                         pIndex=page, pSize=page_size)

    def get_bill_proposers(self, bill_id):
        return self._get('BILLINFOPPSR', BILL_ID=bill_id)

api = AssemblyAPI(API_KEY)
print('Client ready.')

## 1. Collect bills and their proposers

We retrieve bills from a topic area, then for each bill fetch the full
proposer list (lead sponsor + co-sponsors). Each bill that has 2 or more
proposers generates edges in the co-sponsorship graph.

> **Tip:** Use a specific keyword or committee to keep the graph focused.
> A general query returns thousands of bills and takes many minutes.

In [ ]:
import pandas as pd

# --- Configure your query ---
AGE        = '22'          # Assembly number
KEYWORD    = '인공지능'    # Bill keyword (or None for all)
MAX_BILLS  = 100           # Cap to limit runtime (increase for broader analysis)

# Collect bills
all_bills = []
page = 1
while len(all_bills) < MAX_BILLS:
    rows, total = api.search_bills(age=AGE, bill_name=KEYWORD, page=page, page_size=100)
    all_bills.extend(rows)
    print(f'  Page {page}: +{len(rows)} ({len(all_bills)}/{total})', end='\r')
    if len(all_bills) >= total or not rows:
        break
    page += 1
    time.sleep(0.3)

all_bills = all_bills[:MAX_BILLS]
df_bills = pd.DataFrame(all_bills)
print(f'\nCollected {len(df_bills)} bills ({KEYWORD or "all"}, {AGE}th Assembly)')
display(df_bills[['BILL_NO','BILL_NAME','RST_PROPOSER','PROPOSE_DT']].head(8))

In [ ]:
# Fetch proposer lists for each bill
proposers_map = {}   # {BILL_ID: [proposer dicts]}

for i, row in df_bills.iterrows():
    bill_id = row.get('BILL_ID', '')
    if not bill_id:
        continue
    ppsr_rows, _ = api.get_bill_proposers(bill_id=bill_id)
    proposers_map[bill_id] = ppsr_rows
    print(f'  [{i+1}/{len(df_bills)}] {bill_id} — {len(ppsr_rows)} proposers', end='\r')
    time.sleep(0.4)

total_edges = sum(max(0, len(v) - 1) * len(v) // 2 for v in proposers_map.values())
print(f'\nFetched proposers for {len(proposers_map)} bills  |  Potential edges: {total_edges}')

## 2. Build the co-sponsorship graph

**Node**: each legislator who appeared as a proposer on at least one bill.  
**Edge**: two legislators who co-sponsored the same bill.  
**Weight**: the number of bills they co-sponsored together.

In [ ]:
import networkx as nx

def build_cosponsor_graph(bills_df, proposers_map):
    """
    Build undirected co-sponsorship graph.
    Nodes: legislators (attr: party)
    Edges: co-sponsorship links (weight = # shared bills)
    """
    G = nx.Graph()

    for _, bill in bills_df.iterrows():
        bill_id  = bill.get('BILL_ID', '')
        proposers = proposers_map.get(bill_id, [])
        if len(proposers) < 2:
            continue

        members = [
            (p.get('PPSR_NM', ''), p.get('PPSR_POLY_NM', '기타'))
            for p in proposers if p.get('PPSR_NM')
        ]

        for name, party in members:
            if name not in G:
                G.add_node(name, party=party)
            elif not G.nodes[name].get('party'):
                G.nodes[name]['party'] = party

        for i in range(len(members)):
            for j in range(i + 1, len(members)):
                a, _ = members[i]
                b, _ = members[j]
                if G.has_edge(a, b):
                    G[a][b]['weight'] += 1
                else:
                    G.add_edge(a, b, weight=1)

    return G


G = build_cosponsor_graph(df_bills, proposers_map)

print(f'Nodes (legislators): {G.number_of_nodes()}')
print(f'Edges (co-sponsor pairs): {G.number_of_edges()}')
print(f'Density: {nx.density(G):.4f}')
print(f'Connected components: {nx.number_connected_components(G)}')

## 3. Centrality analysis

- **Degree centrality**: how many unique co-sponsors does a legislator have?
- **Weighted degree (strength)**: total number of co-sponsorship events
- **Betweenness centrality**: how often does a legislator lie on the shortest path between two others? (bridges / brokers)
- **Closeness centrality**: how close (in graph distance) is a legislator to all others?

In [ ]:
# Use the largest connected component for centrality calculations
largest_cc = max(nx.connected_components(G), key=len)
G_cc = G.subgraph(largest_cc).copy()
print(f'Largest component: {G_cc.number_of_nodes()} nodes, {G_cc.number_of_edges()} edges')

# Compute centrality measures
degree_c       = nx.degree_centrality(G_cc)
betweenness_c  = nx.betweenness_centrality(G_cc, weight='weight', normalized=True)
closeness_c    = nx.closeness_centrality(G_cc)
strength       = {n: sum(d['weight'] for _, _, d in G_cc.edges(n, data=True)) for n in G_cc.nodes()}

node_data = []
for node in G_cc.nodes():
    node_data.append({
        'name':        node,
        'party':       G_cc.nodes[node].get('party', '기타'),
        'degree':      G_cc.degree(node),
        'strength':    strength[node],
        'degree_c':    round(degree_c[node], 4),
        'between_c':   round(betweenness_c[node], 4),
        'close_c':     round(closeness_c[node], 4),
    })

df_nodes = pd.DataFrame(node_data).sort_values('strength', ascending=False)
print('\nTop legislators by co-sponsorship strength:')
display(df_nodes.head(15))

In [ ]:
import plotly.express as px

PARTY_COLORS = {
    '더불어민주당': '#004EA2',
    '국민의힘':     '#E61E2B',
    '개혁신당':     '#FF7210',
    '조국혁신당':   '#0095DA',
    '진보당':       '#D6001C',
    '무소속':       '#888888',
}

# Top 20 by betweenness (brokers)
top_between = df_nodes.nlargest(20, 'between_c')

fig = px.bar(
    top_between, x='name', y='between_c',
    color='party', color_discrete_map=PARTY_COLORS,
    title=f'Top 20 brokers (betweenness centrality) — {KEYWORD or "all"} bills',
    labels={'name': 'Member', 'between_c': 'Betweenness Centrality', 'party': 'Party'},
)
fig.update_layout(xaxis_tickangle=-45, showlegend=True, height=400)
fig.show()

## 4. Community detection (Louvain method)

The Louvain algorithm finds communities — groups of legislators who co-sponsor
more with each other than with the rest of the network.

We check whether communities align with party boundaries or cut across them.

In [ ]:
try:
    from networkx.algorithms.community import louvain_communities
    communities = louvain_communities(G_cc, weight='weight', seed=42)
    method_used = 'Louvain'
except Exception:
    # Fallback: greedy modularity (available in older networkx)
    from networkx.algorithms.community import greedy_modularity_communities
    communities = list(greedy_modularity_communities(G_cc, weight='weight'))
    method_used = 'Greedy Modularity'

print(f'Community detection method: {method_used}')
print(f'Number of communities: {len(communities)}')

# Assign community id to each node
node_community = {}
for comm_id, comm_nodes in enumerate(communities):
    for n in comm_nodes:
        node_community[n] = comm_id

df_nodes['community'] = df_nodes['name'].map(node_community)

# Party composition of each community
comm_party = (
    df_nodes.groupby(['community', 'party'])
    .size()
    .reset_index(name='count')
    .sort_values(['community', 'count'], ascending=[True, False])
)

print('\nParty composition by community (top 4 communities):')
display(comm_party[comm_party['community'] < 4])

In [ ]:
# Modularity score (how well communities separate from each other)
modularity = nx.algorithms.community.quality.modularity(
    G_cc, communities, weight='weight'
)
print(f'Modularity Q = {modularity:.4f}')
print('  Q > 0.3 suggests meaningful community structure')
print('  Q > 0.5 suggests strong community structure')

## 5. Visualize the network

Nodes are sized by degree and colored by party. Layout uses the Fruchterman-Reingold
force-directed algorithm.

In [ ]:
import plotly.graph_objects as go

# Limit to the largest component and trim isolated nodes for clarity
k = 2.0 / max(G_cc.number_of_nodes() ** 0.5, 1)
pos = nx.spring_layout(G_cc, k=k, iterations=80, seed=42)

# Edge traces (draw first, in background)
edge_traces = []
for a, b, data in G_cc.edges(data=True):
    x0, y0 = pos[a]
    x1, y1 = pos[b]
    w = data.get('weight', 1)
    edge_traces.append(go.Scatter(
        x=[x0, x1, None], y=[y0, y1, None],
        mode='lines',
        line=dict(width=min(w * 0.5 + 0.3, 3.5), color='rgba(150,150,150,0.4)'),
        hoverinfo='none',
        showlegend=False,
    ))

# Node traces grouped by party
party_groups: dict[str, dict] = {}
for node in G_cc.nodes():
    party = G_cc.nodes[node].get('party', '기타')
    deg   = G_cc.degree(node)
    x, y  = pos[node]
    if party not in party_groups:
        party_groups[party] = {'x': [], 'y': [], 'text': [], 'size': []}
    party_groups[party]['x'].append(x)
    party_groups[party]['y'].append(y)
    party_groups[party]['text'].append(
        f'<b>{node}</b><br>Party: {party}<br>Degree: {deg}'
    )
    party_groups[party]['size'].append(max(7, deg * 2.5))

node_traces = [
    go.Scatter(
        x=data['x'], y=data['y'],
        mode='markers',
        name=party,
        hoverinfo='text',
        text=data['text'],
        marker=dict(
            color=PARTY_COLORS.get(party, '#888888'),
            size=data['size'],
            line=dict(width=1, color='white'),
        ),
    )
    for party, data in party_groups.items()
]

fig = go.Figure(
    data=edge_traces + node_traces,
    layout=go.Layout(
        title=f'Co-sponsorship network — "{KEYWORD}" bills, {AGE}th Assembly<br>'
              f'<sup>Nodes: {G_cc.number_of_nodes()}  Edges: {G_cc.number_of_edges()}  '
              f'Modularity Q={modularity:.2f}</sup>',
        showlegend=True,
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=60),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=650,
        paper_bgcolor='white',
    ),
)
fig.show()

## 6. Cross-party collaboration index

For each legislator, what fraction of their co-sponsorships are with members of
other parties? A high value indicates a bridge-builder; low values indicate
within-party co-sponsorship only.

In [ ]:
cross_party_records = []

for node in G_cc.nodes():
    own_party = G_cc.nodes[node].get('party', '')
    total_w   = 0
    cross_w   = 0
    for nbr, data in G_cc[node].items():
        w = data.get('weight', 1)
        nbr_party = G_cc.nodes[nbr].get('party', '')
        total_w += w
        if nbr_party != own_party:
            cross_w += w
    cross_party_records.append({
        'name':        node,
        'party':       own_party,
        'total_cosponsor_w': total_w,
        'cross_party_w':     cross_w,
        'cross_party_pct':   round(cross_w / total_w * 100, 1) if total_w > 0 else 0,
    })

df_cross = pd.DataFrame(cross_party_records).sort_values('cross_party_pct', ascending=False)

print('Top 20 cross-party collaborators:')
display(df_cross.head(20))

In [ ]:
# Distribution of cross-party % by party
avg_cross = (
    df_cross.groupby('party')['cross_party_pct']
    .mean().round(1)
    .reset_index(name='Avg Cross-Party %')
    .sort_values('Avg Cross-Party %', ascending=False)
)

fig = px.bar(
    avg_cross, x='party', y='Avg Cross-Party %',
    color='party', color_discrete_map=PARTY_COLORS,
    title='Average cross-party co-sponsorship by party',
    labels={'party': 'Party'},
    text='Avg Cross-Party %',
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(showlegend=False, xaxis_tickangle=-30, yaxis_range=[0, 105])
fig.show()

## 7. Export datasets

Three export files:

| File | Description |
|---|---|
| `network_nodes.csv` | One row per legislator — centrality, community, cross-party % |
| `network_edges.csv` | One row per co-sponsor pair — weight (# shared bills) |
| `network_bills.csv` | Bill-level data with proposer count |

In [ ]:
# Merge node-level stats
df_nodes_full = (
    df_nodes
    .merge(df_cross[['name','cross_party_w','cross_party_pct']], on='name', how='left')
)
df_nodes_full.to_csv('network_nodes.csv', index=False, encoding='utf-8-sig')

# Edge list
edge_records = [
    {'source': a, 'source_party': G_cc.nodes[a].get('party',''),
     'target': b, 'target_party': G_cc.nodes[b].get('party',''),
     'weight': d.get('weight',1)}
    for a, b, d in G_cc.edges(data=True)
]
df_edges = pd.DataFrame(edge_records)
df_edges.to_csv('network_edges.csv', index=False, encoding='utf-8-sig')

# Bill-level: add proposer count
df_bills['n_proposers'] = df_bills['BILL_ID'].map(
    lambda bid: len(proposers_map.get(bid, []))
)
df_bills.to_csv('network_bills.csv', index=False, encoding='utf-8-sig')

print(f'Saved network_nodes.csv  ({df_nodes_full.shape})')
print(f'Saved network_edges.csv  ({df_edges.shape})')
print(f'Saved network_bills.csv  ({df_bills.shape})')

try:
    from google.colab import files
    for f in ['network_nodes.csv', 'network_edges.csv', 'network_bills.csv']:
        files.download(f)
except Exception:
    pass

---

## Summary

| Concept | Key variable | Interpretation |
|---|---|---|
| **Degree** | `degree` | # unique co-sponsors |
| **Strength** | `strength` | total co-sponsorships (weighted degree) |
| **Betweenness** | `between_c` | brokerage — bridge between clusters |
| **Closeness** | `close_c` | proximity to all others in the network |
| **Community** | `community` | detected cluster (Louvain) |
| **Cross-party %** | `cross_party_pct` | fraction of collaborations outside own party |

**Typical findings in Korean Assembly networks:**
- Party homophily is strong: most edges are within-party
- A handful of brokers connect the two major-party clusters
- Minor-party members show higher cross-party rates (necessity vs. preference)

**References:**
- Fowler, J. H. (2006). Connecting the Congress: A study of cosponsorship networks. *Political Analysis*, 14(4), 456-487.
- Kirkland, J. H., & Gross, J. H. (2014). Measurement and theory in legislative networks. *Social Networks*, 36, 97-116.

**Next steps:**
- Load `network_nodes.csv` + `network_edges.csv` into R (`igraph`) or Python for network regression (ERGM / MRQAP)
- [Back to Tutorial 01: Getting Started](./01_getting_started.ipynb)
- [Back to Tutorial 02: Voting Analysis](./02_voting_analysis.ipynb)